# L17 - Dal codice scritto a mano ai framework: scikit-learn

Questo notebook funziona senza rete in WinPython e in JupyterLite. In JupyterLite, se `import sklearn` produce un errore, eseguire prima una cella con `%pip install scikit-learn`.

Eseguire le celle in ordine con `Maiusc + Invio`.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

def lune(n=200, rumore=0.15, seme=0):
    """Le due lune della lezione L16."""
    rng = np.random.default_rng(seme)
    m = n // 2
    t = rng.uniform(0, np.pi, m)
    a = np.column_stack([np.cos(t), np.sin(t)])
    b = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)])
    X = np.vstack([a, b]) + rng.normal(0, rumore, size=(2 * m, 2))
    y = np.concatenate([np.zeros(m), np.ones(m)]).astype(int)
    return X, y

## 1. La rete della lezione L16 in tre righe

`MLPClassifier` (Multi-Layer Perceptron) è una rete con strati nascosti. I parametri principali:

- `hidden_layer_sizes=(8,)`: un solo strato nascosto con 8 neuroni; `(8, 4)` sono due strati nascosti
- `activation="logistic"`: sigmoide (le altre possibilità sono `"relu"`, `"tanh"`, `"identity"`)
- `solver="sgd"`: discesa del gradiente; `learning_rate_init`: il tasso di apprendimento
- `max_iter`: numero massimo di epoche; `random_state`: seme dei pesi iniziali
- `tol` e `n_iter_no_change`: l'addestramento si ferma prima di `max_iter` se per `n_iter_no_change` epoche consecutive la perdita diminuisce meno di `tol`

`fit(X, y)` addestra, `predict(X)` restituisce le classi previste, `score(X, y)` l'accuratezza.

In [ ]:
X, y = lune()
modello = MLPClassifier(hidden_layer_sizes=(8,), activation="logistic", solver="sgd",
                        learning_rate_init=1.0, max_iter=5000, tol=1e-6,
                        n_iter_no_change=50, random_state=0)
modello.fit(X, y)
print("accuratezza:", modello.score(X, y))
print("forma dei pesi:", [W.shape for W in modello.coefs_])
print("forma dei bias:", [b.shape for b in modello.intercepts_])

I pesi addestrati sono negli attributi `coefs_` e `intercepts_`: sono le matrici `W1.T`, `W2.T` e i vettori `b1`, `b2` della lezione L16 (scikit-learn memorizza le matrici trasposte, con una riga per ingresso).

## 2. La curva della perdita

L'attributo `loss_curve_` contiene la perdita a ogni epoca. scikit-learn usa come perdita l'entropia incrociata (cross-entropy), più adatta dell'MSE ai problemi di classificazione; l'andamento è lo stesso.

In [ ]:
plt.plot(modello.loss_curve_)
plt.xlabel("epoca")
plt.ylabel("perdita")
plt.show()

## 3. Dati di addestramento e dati di verifica

L'accuratezza sugli stessi esempi usati per addestrare è ottimistica: il modello può averli "memorizzati". Per valutarlo onestamente si tiene da parte una parte dei dati, che il modello non vede durante l'addestramento. L'argomento è sviluppato nel corso B.6.

In [ ]:
X_add, X_ver, y_add, y_ver = train_test_split(X, y, test_size=0.3, random_state=0)
modello = MLPClassifier(hidden_layer_sizes=(8,), activation="logistic", solver="sgd",
                        learning_rate_init=1.0, max_iter=5000, tol=1e-6,
                        n_iter_no_change=50, random_state=0)
modello.fit(X_add, y_add)
print("accuratezza sui dati di addestramento:", modello.score(X_add, y_add))
print("accuratezza sui dati di verifica:     ", modello.score(X_ver, y_ver))

## 4. Cifre scritte a mano

scikit-learn contiene un piccolo insieme di 1797 immagini di cifre scritte a mano, di 8 x 8 pixel, con l'etichetta della cifra (0-9). Ogni immagine diventa un vettore di 64 numeri: la rete ha 64 ingressi e 10 uscite, una per cifra.

In [ ]:
from sklearn.datasets import load_digits

cifre = load_digits()
print(cifre.images.shape, cifre.data.shape, cifre.target.shape)

fig, assi = plt.subplots(1, 8, figsize=(10, 1.8))
for i in range(8):
    assi[i].imshow(cifre.images[i], cmap="gray_r")
    assi[i].set_title(str(cifre.target[i]))
    assi[i].axis("off")
plt.show()

In [ ]:
X_c = cifre.data / 16                  # i pixel valgono da 0 a 16: normalizzazione in [0, 1]
Xa, Xv, ya, yv = train_test_split(X_c, cifre.target, test_size=0.3, random_state=0)
rete_cifre = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=0)
rete_cifre.fit(Xa, ya)
print("accuratezza sui dati di verifica:", rete_cifre.score(Xv, yv))

Senza indicare `activation` e `solver` si usano i valori predefiniti: ReLU e l'ottimizzatore Adam, una variante della discesa del gradiente che adatta il tasso di apprendimento durante l'addestramento.

## Esercizi

**Esercizio 1 (base).** Addestrare `MLPClassifier` su XOR (4 esempi) con 4 neuroni nascosti, attivazione `"tanh"`, `solver="lbfgs"` (un metodo adatto a pochi dati) e `random_state=0`. Assegnare le previsioni sui 4 punti a `prev_xor`.

In [ ]:
P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])
m_xor = MLPClassifier(hidden_layer_sizes=(4,), activation="tanh", solver="lbfgs", random_state=0)
m_xor.fit(P, y_xor)
prev_xor = m_xor.predict(P)
print(prev_xor)

In [ ]:
controlla(1, lambda: np.array_equal(prev_xor, [0, 1, 1, 0]))

**Esercizio 2 (base).** Sulle lune (`X_add`, `y_add`, `X_ver`, `y_ver` della sezione 3) confrontare 2, 8 e 32 neuroni nascosti (con gli altri parametri come nella sezione 3). Salvare le accuratezze sui dati di verifica nel dizionario `acc_ver`.

In [ ]:
acc_ver = {}
for k in [2, 8, 32]:
    m = MLPClassifier(hidden_layer_sizes=(k,), activation="logistic", solver="sgd",
                      learning_rate_init=1.0, max_iter=5000, tol=1e-6,
                        n_iter_no_change=50, random_state=0)
    m.fit(X_add, y_add)
    acc_ver[k] = m.score(X_ver, y_ver)
print(acc_ver)

In [ ]:
controlla(2, lambda: set(acc_ver) == {2, 8, 32} and acc_ver[32] >= acc_ver[2])

**Esercizio 3 (standard).** Con le cifre: calcolare le previsioni `prev_cifre` di `rete_cifre` sui dati di verifica e trovare `indici_errati`, gli indici delle immagini classificate male. Mostrare le prime 6 con la cifra vera e quella prevista nel titolo (`Xv[i].reshape(8, 8)` ricostruisce l'immagine).

In [ ]:
prev_cifre = rete_cifre.predict(Xv)
indici_errati = np.where(prev_cifre != yv)[0]
fig, assi = plt.subplots(1, 6, figsize=(9, 1.8))
for a, i in zip(assi, indici_errati[:6]):
    a.imshow(Xv[i].reshape(8, 8), cmap="gray_r")
    a.set_title(f"vera {yv[i]}, prev. {prev_cifre[i]}", fontsize=8)
    a.axis("off")
plt.show()
print(len(indici_errati), "errori su", len(yv))

In [ ]:
controlla(3, lambda: np.array_equal(indici_errati, np.where(rete_cifre.predict(Xv) != yv)[0]))

**Esercizio 4 (standard).** Corrispondenza con la rete scritta a mano. Con il `modello` addestrato sulle lune nella sezione 3, calcolare le probabilità della classe 1 per `X_ver` usando solo NumPy, `coefs_` e `intercepts_`, con la sigmoide in entrambi gli strati: `sigmoide(sigmoide(X @ W1 + b1) @ W2 + b2)`. Confrontare con `modello.predict_proba(X_ver)[:, 1]` e assegnare il risultato a `prob_numpy`.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

W1, W2 = modello.coefs_
b1, b2 = modello.intercepts_
prob_numpy = sigmoide(sigmoide(X_ver @ W1 + b1) @ W2 + b2).ravel()
print(np.round(prob_numpy[:5], 4), np.round(modello.predict_proba(X_ver)[:5, 1], 4))

In [ ]:
controlla(4, lambda: np.allclose(prob_numpy, modello.predict_proba(X_ver)[:, 1]))

**Esercizio 5 (approfondimento).** Sovradattamento (overfitting). Con le lune rumorose (`lune(n=100, rumore=0.35, seme=1)`), divise in addestramento e verifica (30%, `random_state=0`), addestrare una rete con 200 neuroni nascosti e `solver="lbfgs"`, `max_iter=5000`, `random_state=0`. Confrontare l'accuratezza su addestramento (`acc_add_200`) e verifica (`acc_ver_200`). Che cosa significa una grande differenza tra le due?

In [ ]:
Xr, yr = lune(n=100, rumore=0.35, seme=1)
Xra, Xrv, yra, yrv = train_test_split(Xr, yr, test_size=0.3, random_state=0)
grande = MLPClassifier(hidden_layer_sizes=(200,), solver="lbfgs", max_iter=5000, random_state=0)
grande.fit(Xra, yra)
acc_add_200 = grande.score(Xra, yra)
acc_ver_200 = grande.score(Xrv, yrv)
print(acc_add_200, acc_ver_200)

In [ ]:
controlla(5, lambda: acc_add_200 > acc_ver_200)